In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
doctors_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.doctors")

display(doctors_df)

doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,ingestion_timestamp,source_file,load_date
D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02
D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02


In [0]:
print("Total Doctors:", doctors_df.count())

doctors_df.groupBy("doctor_id") \
          .count() \
          .filter(col("count") > 1) \
          .show()

Total Doctors: 10
+---------+-----+
|doctor_id|count|
+---------+-----+
+---------+-----+



In [0]:
doctors_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in doctors_df.columns
]).show()

+---------+----------+---------+--------------+------------+----------------+---------------+-----+-------------------+-----------+---------+
|doctor_id|first_name|last_name|specialization|phone_number|years_experience|hospital_branch|email|ingestion_timestamp|source_file|load_date|
+---------+----------+---------+--------------+------------+----------------+---------------+-----+-------------------+-----------+---------+
|        0|         0|        0|             0|           0|               0|              0|    0|                  0|          0|        0|
+---------+----------+---------+--------------+------------+----------------+---------------+-----+-------------------+-----------+---------+



In [0]:
silver_doctors = (
    doctors_df

    .dropDuplicates(["doctor_id"])

    .withColumn("first_name", initcap(trim(col("first_name"))))

    .withColumn("last_name", initcap(trim(col("last_name"))))

    .withColumn("specialization", initcap(trim(col("specialization"))))

    .withColumn("hospital_branch", initcap(trim(col("hospital_branch"))))

    .withColumn("email", lower(trim(col("email"))))
)

In [0]:
silver_doctors = silver_doctors.filter(
    col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

In [0]:
silver_doctors = silver_doctors.filter(
    length(col("phone_number").cast("string")) == 10
)

In [0]:
silver_doctors = silver_doctors.withColumn(
    "silver_load_timestamp",
    current_timestamp()
)

In [0]:
(
    silver_doctors.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.doctors")
)

In [0]:
display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.doctors")
)

doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,ingestion_timestamp,source_file,load_date,silver_load_timestamp
D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z
D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com,2026-07-02T06:36:40.891Z,doctors.csv,2026-07-02,2026-07-02T07:05:04.656Z


In [0]:
display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.doctors")
)